# Data check

What is in each archive, and why seven Patras participants are left out.
Only participant numbers are printed. The Patras file names also hold
personal details, so they are never shown or saved.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import data
from src.run import load_config

patras = load_config("../configs/patras.yaml", data_root="../data/patras")
pucv = load_config("../configs/pucv.yaml", data_root="../data/pucv")

## Patras, periorbital site

In [ ]:
recs = data.list_recordings(patras)
rows = []
for (s, a), path in recs.items():
    x = data.read_recording(path)
    rows.append(dict(subject=s, acq=a, frames=len(x), shape=f"{x.shape[1]}x{x.shape[2]}",
                     median=float(np.median(x)), max=float(x.max())))
info = pd.DataFrame(rows)

subjects = sorted(info.subject.unique())
print(len(subjects), "participants,", len(info), "recordings")
print("frames per recording:", info.frames.unique().tolist(), " frame size:", info["shape"].unique().tolist())

In [ ]:
grid = info.pivot(index="subject", columns="acq", values="frames")
print("participants missing an acquisition:")
print(grid[grid.isna().any(axis=1)])

In [ ]:
# every file should be rescaled to 0..247, raw sensor files have much larger values
raw = info[info["max"] > 255]
print("recordings stored as raw sensor values:")
print(raw[["subject", "acq", "median", "max"]].to_string(index=False))
print("\nall other recordings: max", info[info["max"] <= 255]["max"].max())

Excluded in `configs/patras.yaml`:

- 20: no 80-minute recording (acquisition 4)
- 11, 12, 13, 14: the sober recording is raw sensor output, so the two
  conditions could be told apart by pixel scale alone
- 9, 24: their files could not be read when the runs were made, so they 
were left out of all runs. Putting them back (run N36) changes MCC by 0.011.

In [ ]:
kept = [s for s in subjects if s not in patras["excluded"]]
print(len(kept), "participants used")

## PUCV-DTF

In [ ]:
recs = data.list_recordings(pucv)
counts = pd.Series({k: len(v) for k, v in recs.items()})
subjects = sorted({s for s, _ in recs})
print(len(subjects), "participants,", len(recs), "recordings,", int(counts.sum()), "images")
print("conditions:", sorted({c for _, c in recs}), " frames per recording:", counts.unique().tolist())
x = data.read_recording(recs[(1, 1)])
print("image size:", x.shape[1:], " value range:", x.min(), "to", x.max())